<a href="https://colab.research.google.com/github/oluwoleadetifa/movie_genre_classification/blob/dev/notebooks/05_bert_text_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!rm -rf /content/movie_genre_classification
!git clone https://github.com/oluwoleadetifa/movie_genre_classification.git
%cd /content/movie_genre_classification

import sys
sys.path.append("/content/movie_genre_classification")

Cloning into 'movie_genre_classification'...
remote: Enumerating objects: 286, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 286 (delta 50), reused 19 (delta 8), pack-reused 185 (from 1)
Receiving objects: 100% (286/286), 1.43 MiB | 6.78 MiB/s, done.
Resolving deltas: 100% (140/140), done.
/content/movie_genre_classification


In [2]:
!pip install transformers torch scikit-learn pandas numpy tqdm

In [3]:
from google.colab import files
uploaded = files.upload()

Saving val.csv to val.csv
Saving train.csv to train.csv
Saving test.csv to test.csv
Saving movies_with_posters.csv to movies_with_posters.csv


In [4]:
import os
import shutil

os.makedirs("/content/movie_genre_classification/data/processed", exist_ok=True)
os.makedirs("/content/movie_genre_classification/data/splits", exist_ok=True)

cwd = os.getcwd()
files_here = os.listdir(cwd)

def move_file(keyword, destination):
    matches = [f for f in files_here if keyword in f]
    if not matches:
        raise FileNotFoundError(f"Could not find file containing: {keyword}")
    shutil.move(os.path.join(cwd, matches[0]), destination)
    print(f"Moved {matches[0]} -> {destination}")

move_file("movies_with_posters", "/content/movie_genre_classification/data/processed/movies_with_posters.csv")
move_file("train", "/content/movie_genre_classification/data/splits/train.csv")
move_file("val", "/content/movie_genre_classification/data/splits/val.csv")
move_file("test", "/content/movie_genre_classification/data/splits/test.csv")

Moved movies_with_posters.csv -> /content/movie_genre_classification/data/processed/movies_with_posters.csv
Moved train.csv -> /content/movie_genre_classification/data/splits/train.csv
Moved val.csv -> /content/movie_genre_classification/data/splits/val.csv
Moved test.csv -> /content/movie_genre_classification/data/splits/test.csv


In [5]:
import pandas as pd

train_df = pd.read_csv("/content/movie_genre_classification/data/splits/train.csv")
val_df = pd.read_csv("/content/movie_genre_classification/data/splits/val.csv")
test_df = pd.read_csv("/content/movie_genre_classification/data/splits/test.csv")

print(train_df.shape, val_df.shape, test_df.shape)
print(train_df.head())

(480, 6) (103, 6) (104, 6)
     movie_id                                        description    genre  \
0  tt20222202  Follows Eden, who goes to a coveted 'Heaven an...   action   
1  tt18411490  In 1999 in London, a teenage Zoya (Katrina Kai...   action   
2  tt15683734  An Indian-American teenager struggling with he...   horror   
3  tt13430858  In London, an award-winning film-maker documen...  romance   
4   tt2488666  Terror strikes when a team of paranormal inves...   horror   

                                          image_path  image_exists  label  
0  /Users/oluwoleadetifa/Library/CloudStorage/Goo...          True      0  
1  /Users/oluwoleadetifa/Library/CloudStorage/Goo...          True      0  
2  /Users/oluwoleadetifa/Library/CloudStorage/Goo...          True      2  
3  /Users/oluwoleadetifa/Library/CloudStorage/Goo...          True      3  
4  /Users/oluwoleadetifa/Library/CloudStorage/Goo...          True      2  


In [6]:
import torch
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

TEXT_COLUMN = "description"
LABEL_COLUMN = "label"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
bert_model = AutoModel.from_pretrained("distilbert-base-uncased").to(device)
bert_model.eval()

Device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

In [7]:
@torch.no_grad()
def encode_texts(texts, batch_size=16, max_length=256):
    embeddings = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = list(texts[i:i + batch_size])

        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoded = {k: v.to(device) for k, v in encoded.items()}
        outputs = bert_model(**encoded)

        hidden = outputs.last_hidden_state
        mask = encoded["attention_mask"].unsqueeze(-1)

        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        embeddings.append(pooled.cpu().numpy())

    return np.vstack(embeddings)

In [8]:
sample_embeddings = encode_texts(train_df[TEXT_COLUMN].tolist()[:8], batch_size=4)
print(sample_embeddings.shape)

100%|██████████| 2/2 [00:03<00:00,  1.99s/it]

(8, 768)


In [9]:
X_train_bert = encode_texts(train_df[TEXT_COLUMN].tolist(), batch_size=16)
X_val_bert = encode_texts(val_df[TEXT_COLUMN].tolist(), batch_size=16)
X_test_bert = encode_texts(test_df[TEXT_COLUMN].tolist(), batch_size=16)

y_train = train_df[LABEL_COLUMN].values
y_val = val_df[LABEL_COLUMN].values
y_test = test_df[LABEL_COLUMN].values

print(X_train_bert.shape, X_val_bert.shape, X_test_bert.shape)

100%|██████████| 7/7 [00:46<00:00,  6.65s/it]

(480, 768) (103, 768) (104, 768)


In [10]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_bert, y_train)

LogisticRegression(max_iter=1000)

In [11]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = clf.predict(X_test_bert)

print("BERT Test Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

BERT Test Accuracy: 0.8173076923076923
              precision    recall  f1-score   support

           0       0.82      0.86      0.84        36
           1       0.57      0.75      0.65        16
           2       0.94      0.89      0.92        38
           3       0.89      0.57      0.70        14

    accuracy                           0.82       104
   macro avg       0.81      0.77      0.78       104
weighted avg       0.84      0.82      0.82       104

Confusion Matrix:
[[31  4  1  0]
 [ 2 12  1  1]
 [ 3  1 34  0]
 [ 2  4  0  8]]


In [12]:
import json
import os

os.makedirs("/content/movie_genre_classification/outputs/metrics", exist_ok=True)

results_bert = {
    "test_accuracy": accuracy_score(y_test, y_pred),
    "classification_report": classification_report(y_test, y_pred, output_dict=True),
    "confusion_matrix": confusion_matrix(y_test, y_pred).tolist()
}

with open("/content/movie_genre_classification/outputs/metrics/bert_results.json", "w") as f:
    json.dump(results_bert, f, indent=2)

print("BERT results saved.")

BERT results saved.
